# Notebook 00b-v2 — regime-specific MiniFASNetV2 5K tuning

Reuses the frozen Stage A–D candidate artifacts after strict provenance checks, then tunes pretrained CSMR in Stages E/F. Tune-Val selects checkpoints and optimizer recipes; LCC is external-development input for CSMR selection. Test-5K is never evaluated and CASIA remains untouched. Mount the old 00b output dataset and set `PRIOR_TUNING_OUTPUT_DIR` if needed.


In [22]:
# CONFIG / explicit Kaggle mounts. Freeze before the first candidate run.
CELEBA_ROOT=("/kaggle/input/datasets/attentionlayer241/celeba-spoof-for-face-antispoofing/"
             "CelebA_Spoof_/CelebA_Spoof")
LCC_FASD_ROOT="/kaggle/input/datasets/aleksandrpikul222/lcc-fasd/LCC_FASD"
FINAL_RESOURCE_DIR="/kaggle/input/datasets/maithanhphuong/final-resources-v1/minifasnet_final_resources_v1"
MINIFASNET_SOURCE_PATH="/kaggle/input/datasets/maithanhphuong/minifasnet/MiniFASNet.py"
MINIFASNET_CHECKPOINT_PATH="/kaggle/input/datasets/maithanhphuong/minifasnet/2.7_80x80_MiniFASNetV2.pth"
OUTPUT_ROOT="/kaggle/working/minifasnetv2_5k_hparam_tuning_v2"
# Exact old 00b output directory. Change this only if the old run is mounted elsewhere.
PRIOR_TUNING_OUTPUT_DIR="/kaggle/input/datasets/maithanhphuong/priortuning/minifasnetv2_5k_hparam_tuning"
TUNE_DATA_SEED=43; TRAIN_SEED=100
BATCH_SIZE=128 # If OOM, restart ALL candidates from clean output with 64.
NUM_WORKERS=4; AMP=True; RESUME=True
MAX_EPOCHS=6; EARLY_STOP_PATIENCE=2; EARLIEST_EARLY_STOP_EPOCH=3
WEIGHT_DECAY=1e-4; LABEL_SMOOTHING=.1; GRAD_CLIP=5.
METRIC_TIE_EPS=1e-12; METRIC_VERIFY_EPS=1e-6
VAL_ACER_MAX_REGRESSION_PP=1.0
CROP_SCALE=2.7; INPUT_SIZE=80
CSMR_WARMUP_EPOCHS=1; CSMR_BAND_CENTERS=[.15,.45,.75]
CSMR_BAND_SIGMA=.10; CSMR_GAIN_RANGE=[.65,.90]; CSMR_PRESERVE_DC=True
SELECTION_CHUNK_SIZE=32
SCRATCH_LR_CANDIDATES=[1e-4,3e-4,1e-3]
PRETRAINED_LR_CANDIDATES=[{'backbone_lr':5e-5,'head_lr':5e-4},
                          {'backbone_lr':1e-4,'head_lr':1e-3},
                          {'backbone_lr':2e-4,'head_lr':1e-3}]
CSMR_LAMBDA_CANDIDATES=[.10,.20,.40]
CSMR_GAIN_CANDIDATES=[('mild',[.75,.95]),('current',[.65,.90]),('strong',[.55,.85])]
assert len(SCRATCH_LR_CANDIDATES)==len(PRETRAINED_LR_CANDIDATES)==len(CSMR_LAMBDA_CANDIDATES)==len(CSMR_GAIN_CANDIDATES)==3

PRETRAINED_CSMR_LAMBDA_CANDIDATES=list(CSMR_LAMBDA_CANDIDATES)
PRETRAINED_CSMR_GAIN_CANDIDATES=[(name,list(gain)) for name,gain in CSMR_GAIN_CANDIDATES]


In [23]:
import csv, hashlib, importlib.util, json, math, os, random, shutil, time, zipfile, copy, gc
from collections import Counter, OrderedDict
from pathlib import Path, PurePosixPath
import cv2, numpy as np, pandas as pd, torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.metrics import confusion_matrix, roc_curve
from IPython.display import FileLink, display
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def sha256_text(value):
    return hashlib.sha256(value.encode("utf-8")).hexdigest()


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def ordered_path_fingerprint(paths):
    return sha256_text("\n".join(map(str, paths)))


def assert_ordered_keys(actual, expected, name):
    actual = list(map(str, actual))
    expected = list(map(str, expected))
    if actual != expected:
        raise RuntimeError(
            "{} sample keys/order differ from the frozen manifest: actual_n={}, expected_n={}".format(
                name, len(actual), len(expected)
            )
        )


def bbox_jitter(record):
    x1, y1, x2, y2 = map(float, record["bbox_xyxy"])
    w, h = x2 - x1, y2 - y1
    if random.random() < 0.20:
        base = max(w, h)
        cx, cy = (x1 + x2) / 2.0, (y1 + y2) / 2.0
        scale = random.uniform(0.95, 1.05)
        cx += random.uniform(-0.05, 0.05) * base
        cy += random.uniform(-0.05, 0.05) * base
        w, h = w * scale, h * scale
        return [cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2]
    return [x1, y1, x2, y2]


def apply_e1_mild_augmentation(rgb):
    if random.random() < 0.5:
        rgb = cv2.flip(rgb, 1)
    if random.random() < 0.30:
        alpha = random.uniform(0.90, 1.10)
        beta = random.uniform(-0.10, 0.10) * 255.0
        rgb = np.clip(rgb.astype(np.float32) * alpha + beta, 0, 255).astype(np.uint8)
    return rgb


def binary_score(logits):
    logits = torch.as_tensor(logits)
    if logits.ndim != 2 or logits.shape[1] != 3:
        raise ValueError("PAD logits must have shape [N, 3] for Real/Physical/Digital")
    if not torch.isfinite(logits).all():
        raise ValueError("PAD logits contain NaN or Infinity")
    return logits[:, 0] - torch.logsumexp(logits[:, 1:], dim=1)


def safe_auc(y_real, scores):
    from sklearn.metrics import roc_auc_score
    y_real = np.asarray(y_real, dtype=np.int64)
    scores = np.asarray(scores, dtype=np.float64)
    if y_real.ndim != 1 or scores.ndim != 1 or len(y_real) != len(scores) or len(y_real) == 0:
        raise ValueError("AUC needs non-empty one-dimensional labels and scores of equal length")
    if not np.isfinite(scores).all():
        raise ValueError("AUC scores contain NaN or Infinity")
    if set(map(int, np.unique(y_real))) != {0, 1}:
        raise ValueError("AUC requires both binary classes (positive class is Real)")
    return float(roc_auc_score(y_real, scores))


def metrics_binary(y3, scores, threshold, coverage=None, candidate_n=None):
    y3 = np.asarray(y3, dtype=np.int64)
    scores = np.asarray(scores, dtype=np.float64)
    if y3.ndim != 1 or scores.ndim != 1 or len(y3) != len(scores) or len(y3) == 0:
        raise ValueError("PAD metrics need non-empty 1-D labels/scores with equal lengths")
    if not set(map(int, np.unique(y3))).issubset({0, 1, 2}):
        raise ValueError("PAD labels must be from the frozen class mapping 0/1/2")
    if not np.isfinite(scores).all() or not np.isfinite(float(threshold)):
        raise ValueError("PAD scores and threshold must be finite")
    y_real = (y3 == 0).astype(np.int64)
    if set(map(int, np.unique(y_real))) != {0, 1}:
        raise ValueError("PAD metrics require at least one Real and one Attack sample")
    if coverage is not None and (not np.isfinite(float(coverage)) or not 0.0 <= float(coverage) <= 1.0):
        raise ValueError("Detector coverage must be finite and within [0, 1]")
    scored_n = len(y3)
    candidate_n = int(candidate_n if candidate_n is not None else scored_n)
    if candidate_n < scored_n:
        raise ValueError("candidate_n cannot be less than n_scored")
    pred_real = scores >= float(threshold)
    real = y_real == 1
    attack = y_real == 0
    bpcer = float(np.mean(~pred_real[real]))
    apcer = float(np.mean(pred_real[attack]))
    acer = 0.5 * (apcer + bpcer)
    return {
        "n_scored": int(scored_n),
        "candidate_n": candidate_n,
        "detector_coverage": float(coverage if coverage is not None else 1.0),
        "apcer": apcer,
        "bpcer": bpcer,
        "acer": acer,
        "hter": acer,
        "auc": safe_auc(y_real, scores),
        "accuracy": float(np.mean(pred_real == y_real)),
        "locked_logit_threshold": float(threshold),
    }


def metrics_celeba(y3, logits, threshold):
    scores = binary_score(torch.as_tensor(logits)).cpu().numpy()
    result = metrics_binary(y3, scores, threshold, coverage=1.0, candidate_n=len(y3))
    result["accuracy_3class"] = float(np.mean(np.argmax(logits, axis=1) == np.asarray(y3)))
    return result, scores


def make_prediction_frame(keys, y3, codes, logits, threshold):
    scores = binary_score(torch.as_tensor(logits)).cpu().numpy()
    pred_real = scores >= threshold
    result = pd.DataFrame({
        "key": list(map(str, keys)),
        "true_class": np.asarray(y3, dtype=np.int64),
        "attack_code": np.asarray(codes, dtype=np.int64),
        "logit_real": logits[:, 0],
        "logit_physical": logits[:, 1],
        "logit_digital": logits[:, 2],
        "pad_logit_score": scores,
        "pred_real": pred_real,
        "correct_binary": pred_real == (np.asarray(y3) == 0),
        "pred_class_3": np.argmax(logits, axis=1),
        "correct_3class": np.argmax(logits, axis=1) == np.asarray(y3),
    })
    return result


def eval_spatial(model, loader, device, amp):
    model.eval()
    logits_all, y_all, keys_all, codes_all = [], [], [], []
    with torch.no_grad():
        for x, y, keys, codes in loader:
            x = x.to(device, non_blocking=True)
            with torch.cuda.amp.autocast(enabled=amp):
                logits = model(x)
            logits_all.append(logits.float().cpu().numpy())
            y_all.extend(np.asarray(y, dtype=np.int64).tolist())
            keys_all.extend(list(keys))
            codes_all.extend(np.asarray(codes, dtype=np.int64).tolist())
    if not logits_all:
        raise RuntimeError("Spatial evaluation loader produced no samples")
    logits = np.concatenate(logits_all)
    labels = np.asarray(y_all, dtype=np.int64)
    codes = np.asarray(codes_all, dtype=np.int64)
    if len(logits) != len(labels) or len(keys_all) != len(labels) or len(codes) != len(labels):
        raise RuntimeError("Spatial evaluation outputs have inconsistent sample counts")
    return logits, labels, list(map(str, keys_all)), codes


def eval_lcc_spatial(model, loader, device, amp, lcc_frame, threshold):
    model.eval()
    logits_all, y_all, keys_all = [], [], []
    with torch.no_grad():
        for x, y, keys, _status in loader:
            x = x.to(device, non_blocking=True)
            with torch.cuda.amp.autocast(enabled=amp):
                logits = model(x)
            logits_all.append(logits.float().cpu().numpy())
            y_all.extend(np.asarray(y, dtype=np.int64).tolist())
            keys_all.extend(list(keys))
    if not logits_all:
        raise RuntimeError("Common LCC VALID evaluation loader produced no samples")
    logits = np.concatenate(logits_all)
    y = np.asarray(y_all, dtype=np.int64)
    keys = list(map(str, keys_all))
    expected_paths = lcc_frame.loc[lcc_frame["status"].astype(str) == "VALID", "path"].astype(str).tolist()
    assert_ordered_keys(keys, expected_paths, "Stage 1 LCC")
    if len(logits) != len(y) or len(keys) != len(y):
        raise RuntimeError("Stage 1 LCC logits/labels/keys have inconsistent counts")
    scores = binary_score(torch.as_tensor(logits)).cpu().numpy()
    scored = pd.DataFrame({
        "path": keys, "label": y,
        "logit_real": logits[:, 0],
        "logit_physical": logits[:, 1],
        "logit_digital": logits[:, 2],
        "pad_logit_score": scores,
    })
    full = lcc_frame[["path", "label", "status"]].merge(
        scored, on=["path", "label"], how="left", validate="one_to_one"
    )
    full["dataset_role"] = "LCC-FASD external-dev / cross-domain stress set"
    valid = full["status"].astype(str).eq("VALID")
    if full.loc[valid, "pad_logit_score"].isna().any() or int(valid.sum()) != len(keys):
        raise RuntimeError("Stage 1 inference failed to score the exact common LCC VALID set")
    full["pred_real"] = np.nan
    full.loc[valid, "pred_real"] = full.loc[valid, "pad_logit_score"] >= float(threshold)
    full["pred_real"] = full["pred_real"].where(valid, np.nan)
    return logits, y, keys, full


def write_zip(folder, zip_path):
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
        for path in sorted(Path(folder).rglob("*")):
            if path.is_file() and path.resolve() != Path(zip_path).resolve():
                archive.write(path, arcname=Path(Path(folder).name) / path.relative_to(folder))


def strict_json_write(path,payload):
    Path(path).write_text(json.dumps(payload,indent=2,allow_nan=False)+'\n')

def strict_json_load(path):
    return json.loads(Path(path).read_text(),parse_constant=lambda s: (_ for _ in ()).throw(ValueError(s)))

def ordered_sha(keys):return sha256_text('\n'.join(map(str,keys)))

def threshold_from_val(y3,scores):
    y=np.asarray(y3,dtype=np.int64);s=np.asarray(scores,dtype=np.float64)
    if len(y)!=len(s) or not len(y) or not np.isfinite(s).all() or set(np.unique(y))!={0,1,2}:
        raise ValueError('Val threshold needs finite 3-class labels/scores')
    order=np.argsort(s,kind='mergesort');v=s[order];real=(y[order]==0)
    nr=int(real.sum());na=len(y)-nr;bound=np.r_[0,np.flatnonzero(np.diff(v))+1,len(v)]
    rp=np.r_[0,np.cumsum(real)];ap=np.r_[0,np.cumsum(~real)];pairs=[]
    for j,k in enumerate(bound):
        t=float(np.nextafter(v[0],-np.inf)) if j==0 else (float(np.nextafter(v[-1],np.inf)) if j==len(bound)-1 else float(v[k-1]/2+v[k]/2))
        pairs.append((.5*(rp[k]/nr+(na-ap[k])/na),abs(t),t))
    return min(pairs)[2]

def resource_tree_sha(root):
    h=hashlib.sha256()
    for p in sorted(Path(root).rglob('*')):
        if p.is_file():h.update(str(p.relative_to(root)).encode()+b'\0'+sha256_file(p).encode()+b'\n')
    return h.hexdigest()
def mini_crop_bgr(image,bbox_xyxy,scale=CROP_SCALE,size=INPUT_SIZE):
    # Official CropImage._get_new_box geometry, adapted from xyxy face coordinates.
    src_h,src_w=image.shape[:2]
    x1,y1,x2,y2=map(float,bbox_xyxy)
    box_w,box_h=x2-x1,y2-y1
    if box_w<=0 or box_h<=0 or src_w<2 or src_h<2:
        raise ValueError('Invalid MiniFASNet image/bbox geometry')
    used_scale=min((src_h-1)/box_h,(src_w-1)/box_w,float(scale))
    new_w,new_h=box_w*used_scale,box_h*used_scale
    center_x,center_y=box_w/2+x1,box_h/2+y1
    left,top=center_x-new_w/2,center_y-new_h/2
    right,bottom=center_x+new_w/2,center_y+new_h/2
    if left<0: right-=left;left=0
    if top<0: bottom-=top;top=0
    if right>src_w-1: left-=right-src_w+1;right=src_w-1
    if bottom>src_h-1: top-=bottom-src_h+1;bottom=src_h-1
    lx,ty,rx,by=map(int,(left,top,right,bottom))
    if lx<0 or ty<0 or rx>=src_w or by>=src_h or rx<lx or by<ty:
        raise RuntimeError(f'MiniFASNet crop escaped image: {(lx,ty,rx,by)}')
    patch=image[ty:by+1,lx:rx+1]
    if patch.size==0:raise RuntimeError('Empty MiniFASNet crop')
    return cv2.resize(patch,(size,size)),(lx,ty,rx,by),used_scale


def mini_tensor(patch):
    if patch.shape!=(INPUT_SIZE,INPUT_SIZE,3) or patch.dtype!=np.uint8:
        raise ValueError('Expected 80x80 uint8 BGR patch')
    return torch.from_numpy(np.ascontiguousarray(patch.transpose(2,0,1))).float().div_(255.)


def source_module(path):
    spec=importlib.util.spec_from_file_location('local_minifasnet_v2',path)
    if spec is None or spec.loader is None:raise RuntimeError('Cannot import local MiniFASNet.py')
    module=importlib.util.module_from_spec(spec);spec.loader.exec_module(module)
    return module


def new_model(module):
    return module.MiniFASNetV2(embedding_size=128,conv6_kernel=(5,5),
        drop_p=.2,num_classes=3,img_channel=3)


def state_fingerprint(state):
    h=hashlib.sha256()
    for key in sorted(state):
        value=state[key].detach().cpu().contiguous()
        h.update(key.encode());h.update(str(value.dtype).encode());h.update(str(tuple(value.shape)).encode())
        h.update(value.numpy().tobytes())
    return h.hexdigest()


def load_pretrained_canonical(module,checkpoint_path):
    raw=torch.load(checkpoint_path,map_location='cpu',weights_only=True)
    if not isinstance(raw,OrderedDict) or not raw or not all(k.startswith('module.') for k in raw):
        raise RuntimeError('Expected official OrderedDict with module. prefix')
    stripped=OrderedDict((k[7:],v) for k,v in raw.items())
    if list(stripped['prob.weight'].shape)!=[3,128]:raise RuntimeError('Official prob.weight shape differs from [3,128]')
    if tuple(stripped['conv_6_dw.conv.weight'].shape[-2:])!=(5,5):raise RuntimeError('Official conv6 kernel is not 5x5')
    seed_everything(TRAIN_SEED)
    model=new_model(module)
    model_keys=model.state_dict()
    skipped=['prob.weight']
    shape_mismatch=[k for k,v in stripped.items() if k in model_keys and tuple(v.shape)!=tuple(model_keys[k].shape) and k not in skipped]
    unexpected=[k for k in stripped if k not in model_keys]
    missing_pretrained=[k for k in model_keys if k not in stripped and k not in skipped]
    if shape_mismatch or unexpected or missing_pretrained:raise RuntimeError(f'Unrelated checkpoint mismatch: shape={shape_mismatch[:5]} unexpected={unexpected[:5]} missing={missing_pretrained[:5]}')
    compatible=OrderedDict((k,v) for k,v in stripped.items() if k not in skipped)
    missing,unexpected_load=model.load_state_dict(compatible,strict=False)
    if set(missing)!={'prob.weight'} or unexpected_load:raise RuntimeError(f'Loading audit failed: missing={missing}, unexpected={unexpected_load}')
    model.prob=nn.Linear(128,3,bias=False)  # new task semantics, deterministic seed 100
    if not model.prob.weight.requires_grad or not model.conv1.conv.weight.requires_grad:
        raise RuntimeError('Classifier/backbone must remain trainable')
    audit={'schema':'mini_fasnet_pretrained_loading_audit_v1','checkpoint_file':checkpoint_path.name,
        'checkpoint_sha256':sha256_file(checkpoint_path),'checkpoint_bytes':checkpoint_path.stat().st_size,
        'architecture_source_sha256':sha256_file(MINIFASNET_SOURCE),
        'checkpoint_keys':len(raw),'loaded_pretrained_key_count':len(compatible),
        'missing_keys':list(missing),'unexpected_keys':list(unexpected_load),
        'intentionally_skipped_keys':skipped,'shape_mismatch_keys':shape_mismatch,
        'input_size':[80,80],'crop_scale':CROP_SCALE,'channel_order':'BGR',
        'conv6_kernel':[5,5],'embedding_size':128,'classifier_shape':[3,128],
        'total_params':sum(p.numel() for p in model.parameters()),
        'trainable_params':sum(p.numel() for p in model.parameters() if p.requires_grad)}
    return model,audit
import torch.nn.functional as F
BANDS=("LOW","MID","HIGH")
def normalize_unit(x):
    # MiniFASNet checkpoint expects BGR float [0,1]; no MNV3 mean/std.
    return x


def radial_masks(size=INPUT_SIZE, device='cpu'):
    y = torch.arange(size, device=device, dtype=torch.float32) - size // 2
    yy, xx = torch.meshgrid(y, y, indexing='ij')
    radius = torch.sqrt(xx.square() + yy.square()) / math.sqrt(2 * (size // 2) ** 2)
    centers = torch.as_tensor(CSMR_BAND_CENTERS, dtype=torch.float32, device=device)
    masks = torch.exp(-((radius[None] - centers[:, None, None]) ** 2) /
                      (2 * CSMR_BAND_SIGMA ** 2))
    if masks.shape != (3, size, size) or not torch.isfinite(masks).all() or not ((masks >= 0) & (masks <= 1)).all():
        raise RuntimeError('Invalid CSMR radial masks')
    if len(set(map(float, CSMR_BAND_CENTERS))) != 3:
        raise RuntimeError('CSMR band centers must differ')
    return masks, radius


def spectral_views(x, gains, masks):
    if x.ndim != 4 or x.shape[1:] != (3, INPUT_SIZE, INPUT_SIZE) or not torch.isfinite(x).all():
        raise ValueError('CSMR input must be finite [B,3,80,80]')
    x = x.float()
    gains = gains.to(device=x.device, dtype=torch.float32).reshape(-1)
    if len(gains) != len(x) or not ((gains >= 0) & (gains <= 1)).all():
        raise ValueError('Invalid gains')
    spectrum = torch.fft.fftshift(torch.fft.fft2(x, norm='ortho'), dim=(-2, -1))
    gain_fields = 1 - (1 - gains[:, None, None, None]) * masks[None]
    if CSMR_PRESERVE_DC:
        gain_fields[:, :, INPUT_SIZE // 2, INPUT_SIZE // 2] = 1.0
    perturbed = spectrum[:, None] * gain_fields[:, :, None]
    raw = torch.fft.ifft2(torch.fft.ifftshift(perturbed, dim=(-2, -1)), norm='ortho').real
    clipped = ((raw < 0) | (raw > 1)).float().mean(dim=(2, 3, 4))
    views = raw.clamp(0, 1)
    if not torch.isfinite(views).all():
        raise RuntimeError('Non-finite CSMR intervention')
    return views, gain_fields, clipped


def aligned_margin(logits, labels):
    d = binary_score(logits.float())
    labels = torch.as_tensor(labels, device=d.device)
    if not torch.isin(labels, torch.tensor([0, 1, 2], device=d.device)).all():
        raise ValueError('Invalid PAD label')
    return torch.where(labels == 0, d, -d)


def harmful_margin_loss(clean_margin, worst_margin):
    drop = clean_margin.detach() - worst_margin
    harmful = drop > 0
    per = F.smooth_l1_loss(worst_margin, clean_margin.detach(), reduction='none')
    return (per * harmful.float()).mean(), harmful, drop


def selection_by_margins(margins):
    if margins.ndim != 2 or margins.shape[1] != 3 or not torch.isfinite(margins).all():
        raise ValueError('Expected finite [B,3] margins')
    return margins.argmin(dim=1)


def select_worst_view(model, base, labels, masks, amp, chunk_size):
    if chunk_size < 1:
        raise ValueError('Selection chunk size must be positive')
    was_training = model.training
    model.eval()
    chosen, indexes, all_gains, all_clips = [], [], [], []
    try:
        with torch.no_grad():
            for start in range(0, len(base), chunk_size):
                x = base[start:start+chunk_size]
                y = labels[start:start+chunk_size]
                gains = torch.empty(len(x), device=x.device).uniform_(*CSMR_GAIN_RANGE)
                views, _, clips = spectral_views(x, gains, masks)
                margins = []
                for band in range(3):
                    with torch.cuda.amp.autocast(enabled=amp):
                        logits = model(normalize_unit(views[:, band]))
                    margins.append(aligned_margin(logits, y))
                indexes_chunk = selection_by_margins(torch.stack(margins, dim=1))
                ix = torch.arange(len(x), device=x.device)
                chosen.append(views[ix, indexes_chunk].detach())
                indexes.append(indexes_chunk.detach())
                all_gains.append(gains.detach())
                all_clips.append(clips[ix, indexes_chunk].detach())
    finally:
        model.train(was_training)
    return tuple(torch.cat(a) for a in (chosen, indexes, all_gains, all_clips))


def intervention_sanity(masks, device):
    gen = torch.Generator(device=device).manual_seed(TRAIN_SEED)
    x = torch.rand((2, 3, INPUT_SIZE, INPUT_SIZE), generator=gen, device=device)
    gain = torch.tensor([0.65, 0.90], device=device)
    views, fields, clips = spectral_views(x, gain, masks)
    assert views.shape == (2, 3, 3, INPUT_SIZE, INPUT_SIZE)
    assert torch.all(fields[:, :, INPUT_SIZE//2, INPUT_SIZE//2] == 1)
    unity, _, _ = spectral_views(x, torch.ones(2, device=device), masks)
    assert torch.max(torch.abs(unity[:, 0] - x)).item() < 1e-5
    ft = torch.fft.fftshift(torch.fft.fft2(x, norm='ortho'), dim=(-2,-1))
    modified = ft[:, None] * fields[:, :, None]
    valid = ft.abs()[:, None] > 1e-5
    phase_error = (torch.angle(modified) - torch.angle(ft[:, None])).abs()
    phase_error = torch.minimum(phase_error, 2*math.pi-phase_error)
    assert phase_error.expand_as(valid.expand_as(modified))[valid.expand_as(modified)].max().item() < 1e-4
    report = {}
    for i, band in enumerate(BANDS):
        diff = (views[:, i] - x).abs()
        report[band] = {'mean_absolute_pixel_difference': float(diff.mean()),
                        'max_absolute_pixel_difference': float(diff.max()),
                        'clipped_pixel_fraction': float(clips[:, i].mean())}
        if report[band]['clipped_pixel_fraction'] > 0.05:
            print('WARNING: clipping exceeds 5% for', band)
    return report
class FinalCelebADataset(Dataset):
    def __init__(self,frame,root,records,train=False):
        self.frame=frame.reset_index(drop=True);self.root=Path(root);self.records=records;self.train=train
        missing=set(self.frame.sample_key.astype(str))-set(records)
        if missing:raise RuntimeError(f'CelebA bbox missing {len(missing)} manifest keys')
    def __len__(self):return len(self.frame)
    def __getitem__(self,i):
        row=self.frame.iloc[i];key=str(row.sample_key);rec=self.records[key]
        if rec.get('status') not in ('VALID','FALLBACK_DET_FAIL','FALLBACK_SUSPICIOUS') or 'crop_factor' in rec:
            raise RuntimeError('Invalid localization-only bbox: '+key)
        image=cv2.imread(str(self.root/key),cv2.IMREAD_COLOR)
        if image is None:raise FileNotFoundError(self.root/key)
        bbox=bbox_jitter(rec) if self.train else rec['bbox_xyxy']
        patch,_,_=mini_crop_bgr(image,bbox)
        if self.train:patch=apply_e1_mild_augmentation(patch)
        return mini_tensor(patch),int(row.three_class_label),key,int(row.attack_code)

class FinalLCCDataset(Dataset):
    def __init__(self,frame,root,records):
        self.rows=[];self.root=Path(root)
        for row in frame.itertuples(index=False):
            if str(row.status)=='VALID':
                rec=records[str(row.path)]
                if rec.get('status')!='VALID' or 'crop_factor' in rec:raise RuntimeError('LCC localization record invalid')
                self.rows.append((str(row.path),int(row.label),rec))
    def __len__(self):return len(self.rows)
    def __getitem__(self,i):
        key,label,rec=self.rows[i];image=cv2.imread(str(self.root/key),cv2.IMREAD_COLOR)
        if image is None:raise FileNotFoundError(self.root/key)
        patch,_,_=mini_crop_bgr(image,rec['bbox_xyxy'])
        return mini_tensor(patch),label,key,'VALID'


In [24]:
# Bring the completed v1 Stage A-D artifacts into a separate v2 output tree.
# An independent Kaggle session must mount the exact old 00b output and set PRIOR_TUNING_OUTPUT_DIR above.
prior=Path(PRIOR_TUNING_OUTPUT_DIR).expanduser()
output=Path(OUTPUT_ROOT)
EXPECTED_V1_SHA256='2aedb0477c6d0ed05c9a96ba8918497b312fdebd0f820c1170e4c2b56398155e'
if not (prior/'selection/frozen_hparams.json').is_file():
    raise FileNotFoundError('Mount the completed 00b output and set PRIOR_TUNING_OUTPUT_DIR to its exact minifasnetv2_5k_hparam_tuning directory: '+str(prior))
if sha256_file(prior/'selection/frozen_hparams.json')!=EXPECTED_V1_SHA256:
    raise RuntimeError('Prior Stage A-D frozen config SHA256 differs')
if output.exists():
    if not (output/'selection/frozen_hparams.json').is_file():
        raise RuntimeError('Existing v2 output is incomplete or foreign; move it before rerunning')
    if sha256_file(output/'selection/frozen_hparams.json')!=EXPECTED_V1_SHA256:
        raise RuntimeError('Existing v2 output has a different historical frozen config')
else:
    shutil.copytree(prior,output)
print('Verified historical Stage A-D artifacts:',output)


Verified historical Stage A-D artifacts: /kaggle/working/minifasnetv2_5k_hparam_tuning_v2


In [25]:
root=Path(FINAL_RESOURCE_DIR).expanduser()
if not (root/'config.json').is_file():
    raise FileNotFoundError('Set FINAL_RESOURCE_DIR to exact frozen resource directory: '+str(root))
output=Path(OUTPUT_ROOT);tuning_dir=output/'tuning_resources';selection_dir=output/'selection'
for folder in (tuning_dir,selection_dir):folder.mkdir(parents=True,exist_ok=True)
celeba_root=Path(CELEBA_ROOT);lcc_root=Path(LCC_FASD_ROOT)
MINIFASNET_SOURCE=Path(MINIFASNET_SOURCE_PATH)
MINIFASNET_CHECKPOINT=Path(MINIFASNET_CHECKPOINT_PATH)
required=[root/'config.json',root/'provenance.json',root/'manifests/manifest_fingerprints.json',
    root/'manifests/final_train100k.csv',root/'manifests/final_val15k.csv',
    root/'manifests/final_test5k.csv',root/'manifests/lcc_external_dev.csv',
    root/'bbox/celeba_final_bbox_cache.json',root/'bbox/lcc_bbox_cache.json',
    root/'bbox/train_filter_audit.json',MINIFASNET_SOURCE,MINIFASNET_CHECKPOINT]
missing=[str(p) for p in required if not p.is_file()]
if missing:raise FileNotFoundError('Frozen input missing: '+', '.join(missing))
if not celeba_root.is_dir() or not lcc_root.is_dir():raise FileNotFoundError('Set exact CelebA/LCC roots')
resource_config=strict_json_load(root/'config.json');resource_provenance=strict_json_load(root/'provenance.json')
manifest_fp=strict_json_load(root/'manifests/manifest_fingerprints.json')
if resource_config['input']!={'scale':2.7,'size':[80,80],'channels':'BGR','range':'[0,1]'}:
    raise RuntimeError('Frozen MiniFASNet preprocessing changed')
for name in ('final_train100k','final_val15k','final_test5k'):
    p=root/'manifests'/(name+'.csv')
    if sha256_file(p)!=manifest_fp[name]['file_sha256']:raise RuntimeError('Frozen manifest SHA mismatch: '+name)
for path,expected in [(root/'bbox/celeba_final_bbox_cache.json',resource_provenance['celeba_bbox_sha256']),
    (root/'bbox/lcc_bbox_cache.json',resource_provenance['lcc_bbox_sha256']),
    (root/'manifests/lcc_external_dev.csv',resource_provenance['lcc_manifest_sha256']),
    (root/'bbox/train_filter_audit.json',resource_provenance['train_filter_audit_sha256'])]:
    if sha256_file(path)!=expected:raise RuntimeError('Frozen resource SHA mismatch: '+str(path))
# The official Test manifest is checked for bundle integrity above; its rows are not opened.
train_full=pd.read_csv(root/'manifests/final_train100k.csv')
val_full=pd.read_csv(root/'manifests/final_val15k.csv')
assert len(train_full)==100000 and len(val_full)==15000
for name,frame in [('final_train100k',train_full),('final_val15k',val_full)]:
    if ordered_sha(frame.sample_key.astype(str))!=manifest_fp[name]['ordered_sample_key_sha256']:
        raise RuntimeError('Ordered source manifest keys differ: '+name)
assert set(train_full.sample_key).isdisjoint(val_full.sample_key)
assert set(train_full.subject_id.astype(str)).isdisjoint(set(val_full.subject_id.astype(str)))

from sklearn.model_selection import train_test_split
def nested_stratified(frame,n,seed):
    idx=np.arange(len(frame));selected,_=train_test_split(idx,train_size=n,
        stratify=frame.three_class_label.astype(int).to_numpy(),random_state=seed)
    chosen=frame.iloc[selected].reset_index(drop=True).copy()
    chosen['tune_sampling_seed']=seed
    return chosen
tune_train=nested_stratified(train_full,5000,TUNE_DATA_SEED)
tune_val=nested_stratified(val_full,2000,TUNE_DATA_SEED)
assert len(tune_train)==5000 and len(tune_val)==2000
assert set(tune_train.sample_key)<=set(train_full.sample_key)
assert set(tune_val.sample_key)<=set(val_full.sample_key)
assert set(tune_train.subject_id.astype(str)).isdisjoint(set(tune_val.subject_id.astype(str)))
tune_fp={}
for name,frame,source_name in [('tune_train5k',tune_train,'final_train100k'),
                               ('tune_val2k',tune_val,'final_val15k')]:
    if set(frame.three_class_label.astype(int))!={0,1,2}:raise RuntimeError('Tuning class missing')
    path=tuning_dir/(name+'.csv');csv_text=frame.to_csv(index=False)
    if path.exists() and path.read_text()!=csv_text:raise RuntimeError('Previously frozen tuning subset differs: '+name)
    path.write_text(csv_text)
    tune_fp[name]={'sample_count':len(frame),
        'class_counts':{str(k):int(v) for k,v in frame.three_class_label.value_counts().items()},
        'ordered_sample_key_sha256':ordered_sha(frame.sample_key.astype(str)),
        'source_manifest_sha256':manifest_fp[source_name]['file_sha256'],
        'sampling_seed':TUNE_DATA_SEED,'file_sha256':sha256_file(path)}
fp_path=tuning_dir/'tune_manifest_fingerprints.json'
if fp_path.exists() and strict_json_load(fp_path)!=tune_fp:raise RuntimeError('Tuning fingerprint drift')
strict_json_write(fp_path,tune_fp)

bbox_cache=strict_json_load(root/'bbox/celeba_final_bbox_cache.json')['records']
selected_keys=set(tune_train.sample_key.astype(str))|set(tune_val.sample_key.astype(str))
if not selected_keys<=set(bbox_cache):raise RuntimeError('Tuning keys missing from frozen bbox cache')
for key in tune_train.sample_key.astype(str):
    rec=bbox_cache[key]
    if rec.get('status') not in ('VALID','FALLBACK_DET_FAIL'):
        raise RuntimeError('Train hygiene status failed: '+key)
    x1,y1,x2,y2=map(float,rec['bbox_xyxy'])
    if min(x2-x1,y2-y1)<48:raise RuntimeError('Train hygiene face size failed: '+key)
if any('crop_factor' in bbox_cache[k] for k in selected_keys):raise RuntimeError('BBox cache is not localization-only')
for frame in (tune_train,tune_val):
    if not all((celeba_root/k).is_file() for k in frame.sample_key.astype(str)):
        raise FileNotFoundError('Tuning image missing under CELEBA_ROOT')
lcc_frame=pd.read_csv(root/'manifests/lcc_external_dev.csv',keep_default_na=False)
lcc_cache=strict_json_load(root/'bbox/lcc_bbox_cache.json')['records']
if set(lcc_frame.path.astype(str))!=set(lcc_cache):raise RuntimeError('Frozen LCC keys differ')
if set(lcc_frame[lcc_frame.status=='VALID'].label.astype(int))!={0,1}:
    raise RuntimeError('LCC needs Real and Attack')
if not all((lcc_root/k).is_file() for k in lcc_frame.path.astype(str)):
    raise FileNotFoundError('LCC image missing under LCC_FASD_ROOT')
resource_hashes={'bundle_tree_sha256':resource_tree_sha(root),
    'tune_train_manifest_sha256':tune_fp['tune_train5k']['file_sha256'],
    'tune_val_manifest_sha256':tune_fp['tune_val2k']['file_sha256'],
    'celeba_bbox_sha256':sha256_file(root/'bbox/celeba_final_bbox_cache.json'),
    'lcc_manifest_sha256':sha256_file(root/'manifests/lcc_external_dev.csv'),
    'lcc_bbox_sha256':sha256_file(root/'bbox/lcc_bbox_cache.json'),
    'minifasnet_source_sha256':sha256_file(MINIFASNET_SOURCE),
    'official_checkpoint_sha256':sha256_file(MINIFASNET_CHECKPOINT)}
strict_json_write(tuning_dir/'resource_fingerprints.json',resource_hashes)
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu');use_amp=bool(AMP and device.type=='cuda')
print('Frozen tuning resources:',tune_fp,'LCC candidates:',len(lcc_frame),'device:',device)


Frozen tuning resources: {'tune_train5k': {'sample_count': 5000, 'class_counts': {'1': 2315, '0': 1637, '2': 1048}, 'ordered_sample_key_sha256': '52425728967f3a4e62b2915595a45a53886f07d646389a084884c65eefe7b659', 'source_manifest_sha256': '3ec46d1cc6b9bb384f244ba4fcc3657fd1e6f050b13b00b37818a913b590c8b6', 'sampling_seed': 43, 'file_sha256': '33142055672456ea95de978d8af0ebb8d14ae45d65a4d35117ac8eff21e3353f'}, 'tune_val2k': {'sample_count': 2000, 'class_counts': {'1': 922, '0': 656, '2': 422}, 'ordered_sample_key_sha256': '5ade0cea2cc0d339f4a6d7ac477c971ddc057bf1e27931b9219e6a568fd2ff7e', 'source_manifest_sha256': '5adf32016920e2e886e148eaedd6abdc73f092525c070ced2ca476c24326d4a9', 'sampling_seed': 43, 'file_sha256': '8368e1b20411174b0c783a42db0720b64dcd4a30ad4ba720e4e67b6f5afbf1bc'}} LCC candidates: 3766 device: cuda


In [26]:
seed_everything(TRAIN_SEED)
module=source_module(MINIFASNET_SOURCE)
scratch_model=new_model(module)
with torch.no_grad():
    if tuple(scratch_model.eval()(torch.zeros(2,3,80,80)).shape)!=(2,3):raise RuntimeError('MiniFASNet logits contract failed')
scratch_state={k:v.detach().cpu().clone() for k,v in scratch_model.state_dict().items()}
scratch_sha=state_fingerprint(scratch_state)
strict_json_write(tuning_dir/'initialization_fingerprints.json',{'scratch_sha256':scratch_sha,
    'train_seed':TRAIN_SEED,
    'parameter_count':sum(p.numel() for p in scratch_model.parameters())})
masks,_=radial_masks(INPUT_SIZE,device)
sanity=intervention_sanity(masks,device)
assert selection_by_margins(torch.tensor([[2.,-1.,0.]])).item()==1
assert harmful_margin_loss(torch.tensor([0.]),torch.tensor([1.]))[0].item()==0
assert harmful_margin_loss(torch.tensor([1.]),torch.tensor([0.]))[0].item()>0
probe=new_model(module).to(device).train()
before={k:v.clone() for k,v in probe.state_dict().items() if 'running_' in k or 'num_batches_tracked' in k}
select_worst_view(probe,torch.rand(2,3,80,80,device=device),torch.tensor([0,1],device=device),masks,use_amp,2)
assert probe.training and all(torch.equal(v,probe.state_dict()[k]) for k,v in before.items())
del probe,scratch_model
print('Canonical scratch SHA:',scratch_sha,'CSMR sanity:',sanity)


/tmp/ipykernel_58/1261348889.py:414: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=amp):


Canonical scratch SHA: f134adaf0d46b5fb50325aefe05220d2ec6c6b50dcfe34b70d0a58b82c711828 CSMR sanity: {'LOW': {'mean_absolute_pixel_difference': 0.014951439574360847, 'max_absolute_pixel_difference': 0.11515884101390839, 'clipped_pixel_fraction': 0.004140624776482582}, 'MID': {'mean_absolute_pixel_difference': 0.026369640603661537, 'max_absolute_pixel_difference': 0.18637901544570923, 'clipped_pixel_fraction': 0.0009374999790452421}, 'HIGH': {'mean_absolute_pixel_difference': 0.02633221447467804, 'max_absolute_pixel_difference': 0.17739760875701904, 'clipped_pixel_fraction': 0.001015624962747097}}


In [27]:
run_registry={}
def candidate_config(initialization,initial_sha,backbone_lr,classifier_lr,csmr_lambda,gain_range):
    return {'schema':'minifasnetv2_5k_candidate_v1','resource_hashes':resource_hashes,
        'initialization':initialization,'initial_state_sha256':initial_sha,
        'train_seed':TRAIN_SEED,'tune_data_seed':TUNE_DATA_SEED,
        'train_manifest_sha256':tune_fp['tune_train5k']['file_sha256'],
        'val_manifest_sha256':tune_fp['tune_val2k']['file_sha256'],
        'model':{'embedding_size':128,'conv6_kernel':[5,5],'drop_p':.2,'num_classes':3,'img_channel':3},
        'input':{'crop_scale':2.7,'size':[80,80],'color':'BGR','range':'[0,1]'},
        'optimizer':'AdamW','backbone_lr':float(backbone_lr),'classifier_lr':float(classifier_lr),
        'weight_decay':WEIGHT_DECAY,'label_smoothing':LABEL_SMOOTHING,'grad_clip':GRAD_CLIP,
        'scheduler':None,'batch_size':BATCH_SIZE,'amp':use_amp,'max_epochs':MAX_EPOCHS,
        'early_stop_patience':EARLY_STOP_PATIENCE,'earliest_stop_epoch':EARLIEST_EARLY_STOP_EPOCH,
        'augmentation':'same E1 bbox jitter .2, flip .5, brightness/contrast .3',
        'csmr':None if csmr_lambda is None else {'lambda':float(csmr_lambda),'gain_range':list(map(float,gain_range)),
            'band_centers':CSMR_BAND_CENTERS,'band_sigma':CSMR_BAND_SIGMA,
            'warmup_epochs':CSMR_WARMUP_EPOCHS,'preserve_dc':CSMR_PRESERVE_DC}}

def tune_loader(frame,train=False,epoch=0):
    dataset=FinalCelebADataset(frame,celeba_root,bbox_cache,train=train)
    gen=torch.Generator().manual_seed(TRAIN_SEED+epoch)
    def worker_init(worker_id):
        value=TRAIN_SEED+epoch*1000+worker_id
        random.seed(value);np.random.seed(value);torch.manual_seed(value)
    return DataLoader(dataset,batch_size=BATCH_SIZE,shuffle=train,drop_last=False,
        num_workers=NUM_WORKERS,pin_memory=device.type=='cuda',generator=gen,worker_init_fn=worker_init)

def run_candidate(stage,candidate_name,initialization,backbone_lr,classifier_lr,csmr_lambda=None,gain_range=None):
    global CSMR_GAIN_RANGE
    folder=output/stage/candidate_name;folder.mkdir(parents=True,exist_ok=True)
    canonical_state=scratch_state if initialization=='scratch' else pretrained_state
    initial_sha=scratch_sha if initialization=='scratch' else pretrained_sha
    conf=candidate_config(initialization,initial_sha,backbone_lr,classifier_lr,csmr_lambda,gain_range)
    conf_hash=sha256_text(json.dumps(conf,sort_keys=True,separators=(',',':')))
    conf_saved={**conf,'config_hash':conf_hash}
    config_path=folder/'config.json';metrics_path=folder/'metrics.json';best_path=folder/'best.pth';last_path=folder/'last_state.pth'
    if config_path.exists():
        if strict_json_load(config_path)!=conf_saved:raise RuntimeError('Incompatible candidate directory: '+str(folder))
    else:strict_json_write(config_path,conf_saved)
    if conf_hash in run_registry:
        prior=run_registry[conf_hash]
        strict_json_write(folder/'reuse_reference.json',{'config_hash':conf_hash,'source_dir':prior['candidate_dir']})
        return {**prior,'stage':stage,'candidate_name':candidate_name,'reused_or_new':'REUSED'}
    if metrics_path.exists() or best_path.exists() and not last_path.exists():
        if not metrics_path.exists() or not best_path.exists():raise RuntimeError('Incomplete candidate artifacts: '+str(folder))
        saved=strict_json_load(metrics_path)
        if (saved['config_hash']!=conf_hash or saved['resource_hashes']!=resource_hashes or
            saved['checkpoint_sha256']!=sha256_file(best_path) or
            saved['initialization_sha256']!=initial_sha):
            raise RuntimeError('Completed candidate provenance mismatch: '+str(folder))
        result={**saved,'stage':stage,'candidate_name':candidate_name,'candidate_dir':str(folder),'reused_or_new':'REUSED'}
        run_registry[conf_hash]=result
        return result
    seed_everything(TRAIN_SEED)
    model=new_model(module);model.load_state_dict(canonical_state,strict=True)
    if state_fingerprint(model.state_dict())!=initial_sha:raise RuntimeError('Canonical candidate initialization drift')
    model=model.to(device)
    if initialization=='pretrained':
        backbone=[p for name,p in model.named_parameters() if not name.startswith('prob.')]
        groups=[{'params':backbone,'lr':backbone_lr},
                {'params':list(model.prob.parameters()),'lr':classifier_lr}]
    else:groups=[{'params':model.parameters(),'lr':backbone_lr}]
    optimizer=torch.optim.AdamW(groups,weight_decay=WEIGHT_DECAY)
    scheduler=None # Existing final notebooks use no scheduler.
    scaler=torch.cuda.amp.GradScaler(enabled=use_amp)
    criterion=nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    history=[];diagnostics=[];class_diagnostics=[];best_epoch=0
    best_acer=float('inf');best_auc=-float('inf');start_epoch=1;stop_reached=False
    if last_path.exists():
        if not RESUME:raise RuntimeError('Incomplete candidate exists; set RESUME=True or clean its directory')
        state=torch.load(last_path,map_location='cpu',weights_only=False)
        if state.get('config_hash')!=conf_hash or state.get('initialization_sha256')!=initial_sha:
            raise RuntimeError('Candidate resume config/initialization mismatch')
        if 'best_auc' not in state:raise RuntimeError('Old resume lacks best_auc; restart candidate')
        model.load_state_dict(state['model_state']);optimizer.load_state_dict(state['optimizer_state'])
        scaler.load_state_dict(state['scaler_state']);history=state['history']
        diagnostics=state['diagnostics'];class_diagnostics=state['class_diagnostics']
        best_epoch=state['best_epoch'];best_acer=state['best_acer'];best_auc=state['best_auc']
        start_epoch=state['epoch']+1;stop_reached=state['stop_reached']
        random.setstate(state['rng_python']);np.random.set_state(state['rng_numpy'])
        torch.set_rng_state(state['rng_torch'])
        if device.type=='cuda' and state['rng_cuda'] is not None:torch.cuda.set_rng_state_all(state['rng_cuda'])
        if not best_path.is_file():raise RuntimeError('Candidate best checkpoint missing on resume')
    CSMR_GAIN_RANGE=list(gain_range) if gain_range is not None else [.65,.90]
    val_loader=tune_loader(tune_val)
    try:
        for epoch in ([] if stop_reached else range(start_epoch,MAX_EPOCHS+1)):
            started=time.monotonic();model.train();train_loader=tune_loader(tune_train,train=True,epoch=epoch)
            losses=[];cls_losses=[];margin_losses=[];clean_ce_values=[];worst_ce_values=[]
            band_counts=torch.zeros(3,dtype=torch.int64);class_bands=torch.zeros((3,3),dtype=torch.int64)
            class_seen=torch.zeros(3,dtype=torch.int64);class_harm=torch.zeros(3,dtype=torch.int64)
            class_drop=torch.zeros(3,dtype=torch.float64)
            csmr_n=harmful_n=0;drop_sum=clean_margin_sum=worst_margin_sum=gain_sum=clip_sum=0.
            print(f'{stage}/{candidate_name} epoch {epoch}/{MAX_EPOCHS}',flush=True)
            for batch_index,(x,y,_,_) in enumerate(train_loader,1):
                x=x.to(device,non_blocking=True);y=y.to(device,non_blocking=True)
                optimizer.zero_grad(set_to_none=True)
                if csmr_lambda is not None and epoch>CSMR_WARMUP_EPOCHS:
                    selected,bands,gains,clips=select_worst_view(model,x,y,masks,use_amp,SELECTION_CHUNK_SIZE)
                    with torch.cuda.amp.autocast(enabled=use_amp):
                        logits=model(torch.cat([x,selected],dim=0));clean_logits,worst_logits=logits.chunk(2)
                        ce_clean=criterion(clean_logits,y);ce_worst=criterion(worst_logits,y)
                        cls_loss=.5*ce_clean+.5*ce_worst
                        clean_margin=aligned_margin(clean_logits,y);worst_margin=aligned_margin(worst_logits,y)
                        margin_loss,harmful,drop=harmful_margin_loss(clean_margin,worst_margin)
                        loss=cls_loss+float(csmr_lambda)*margin_loss
                    n=len(y);csmr_n+=n;harmful_n+=int(harmful.sum().cpu())
                    drop_sum+=float(drop.clamp_min(0).sum().detach().cpu())
                    clean_margin_sum+=float(clean_margin.sum().detach().cpu())
                    worst_margin_sum+=float(worst_margin.sum().detach().cpu())
                    gain_sum+=float(gains.sum().cpu());clip_sum+=float(clips.sum().cpu())
                    clean_ce_values.append(float(ce_clean.detach().cpu()))
                    worst_ce_values.append(float(ce_worst.detach().cpu()))
                    for band in range(3):band_counts[band]+=int((bands==band).sum().cpu())
                    for cls in range(3):
                        mask=y==cls;class_seen[cls]+=int(mask.sum().cpu())
                        class_harm[cls]+=int((harmful&mask).sum().cpu())
                        class_drop[cls]+=float(drop.clamp_min(0)[mask].sum().detach().cpu())
                        for band in range(3):class_bands[cls,band]+=int((mask&(bands==band)).sum().cpu())
                else:
                    with torch.cuda.amp.autocast(enabled=use_amp):
                        logits=model(x);cls_loss=criterion(logits,y);margin_loss=cls_loss.detach()*0;loss=cls_loss
                    clean_ce_values.append(float(cls_loss.detach().cpu()))
                if not torch.isfinite(loss):raise RuntimeError('Non-finite candidate loss')
                scaler.scale(loss).backward();scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(),GRAD_CLIP)
                scaler.step(optimizer);scaler.update()
                losses.append(float(loss.detach().cpu()));cls_losses.append(float(cls_loss.detach().cpu()))
                margin_losses.append(float(margin_loss.detach().cpu()))
                if batch_index==1 or batch_index%max(1,len(train_loader)//8)==0:
                    print('  batch',batch_index,'/',len(train_loader),'mean loss',float(np.mean(losses)),flush=True)
            val_logits,val_y,val_keys,_=eval_spatial(model,val_loader,device,use_amp)
            assert_ordered_keys(val_keys,tune_val.sample_key.astype(str).tolist(),'Tune-Val')
            scores=binary_score(torch.from_numpy(val_logits)).numpy()
            threshold=threshold_from_val(val_y,scores)
            val_metrics,_=metrics_celeba(val_y,val_logits,threshold)
            history.append({'epoch':epoch,'train_loss':float(np.mean(losses)),
                'val_acer':val_metrics['acer'],'val_auc':val_metrics['auc'],
                'val_threshold':threshold,'seconds':time.monotonic()-started})
            if csmr_lambda is not None:
                d={'epoch':epoch,'ce_clean':float(np.mean(clean_ce_values)),
                    'ce_worst':float(np.mean(worst_ce_values)) if worst_ce_values else None,
                    'classification_loss':float(np.mean(cls_losses)),
                    'margin_loss':float(np.mean(margin_losses)),'total_loss':float(np.mean(losses)),
                    'harmful_fraction':harmful_n/csmr_n if csmr_n else None,
                    'mean_harmful_margin_drop':drop_sum/harmful_n if harmful_n else None,
                    'mean_clean_aligned_margin':clean_margin_sum/csmr_n if csmr_n else None,
                    'mean_worst_aligned_margin':worst_margin_sum/csmr_n if csmr_n else None,
                    'mean_sampled_gain':gain_sum/csmr_n if csmr_n else None,
                    'clipping_fraction':clip_sum/csmr_n if csmr_n else None}
                for band in range(3):
                    d['worst_'+BANDS[band].lower()+'_fraction']=int(band_counts[band])/csmr_n if csmr_n else None
                diagnostics.append(d)
                for cls,label in enumerate(('Real','Physical','Digital')):
                    n=int(class_seen[cls]);r={'epoch':epoch,'class':label,'n':n,
                        'harmful_fraction':int(class_harm[cls])/n if n else None,
                        'mean_harmful_margin_drop':float(class_drop[cls])/int(class_harm[cls]) if class_harm[cls] else None}
                    for band in range(3):
                        r['worst_'+BANDS[band].lower()+'_fraction']=int(class_bands[cls,band])/n if n else None
                    class_diagnostics.append(r)
            acer=float(val_metrics['acer']);auc=float(val_metrics['auc'])
            better=acer<best_acer-METRIC_TIE_EPS or (abs(acer-best_acer)<=METRIC_TIE_EPS and auc>best_auc+METRIC_TIE_EPS)
            if better:
                best_acer=acer;best_auc=auc;best_epoch=epoch
                best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
                temp=best_path.with_suffix('.pth.tmp')
                torch.save({'state_dict':best_state,'epoch':epoch,'best_val_acer':acer,
                    'best_val_auc':auc,'config_hash':conf_hash},temp);os.replace(temp,best_path)
            stop_reached=epoch==MAX_EPOCHS or (epoch>=EARLIEST_EARLY_STOP_EPOCH and epoch-best_epoch>=EARLY_STOP_PATIENCE)
            state={'config_hash':conf_hash,'initialization_sha256':initial_sha,'epoch':epoch,
                'model_state':{k:v.detach().cpu().clone() for k,v in model.state_dict().items()},
                'optimizer_state':optimizer.state_dict(),'scheduler_state':scheduler.state_dict() if scheduler else None,
                'scaler_state':scaler.state_dict(),'best_epoch':best_epoch,'best_acer':best_acer,'best_auc':best_auc,
                'history':history,'diagnostics':diagnostics,'class_diagnostics':class_diagnostics,
                'rng_python':random.getstate(),'rng_numpy':np.random.get_state(),
                'rng_torch':torch.get_rng_state(),'rng_cuda':torch.cuda.get_rng_state_all() if device.type=='cuda' else None,
                'stop_reached':stop_reached}
            temp=last_path.with_suffix('.pth.tmp');torch.save(state,temp);os.replace(temp,last_path)
            print('  Tune-Val ACER',acer,'AUC',auc,'best epoch',best_epoch,flush=True)
            if stop_reached:break
    except torch.cuda.OutOfMemoryError as exc:
        raise RuntimeError('OOM: restart every candidate with BATCH_SIZE=64 in clean output') from exc
    if not best_path.is_file():raise RuntimeError('No Tune-Val selected checkpoint: '+str(folder))
    checkpoint=torch.load(best_path,map_location='cpu',weights_only=True)
    if checkpoint.get('config_hash')!=conf_hash or checkpoint.get('epoch')!=best_epoch:
        raise RuntimeError('Best checkpoint provenance differs')
    model.load_state_dict(checkpoint['state_dict'])
    val_logits,val_y,val_keys,val_codes=eval_spatial(model,val_loader,device,use_amp)
    assert_ordered_keys(val_keys,tune_val.sample_key.astype(str).tolist(),'Frozen Tune-Val')
    locked=threshold_from_val(val_y,binary_score(torch.from_numpy(val_logits)).numpy())
    val_metrics,_=metrics_celeba(val_y,val_logits,locked)
    if abs(val_metrics['acer']-best_acer)>METRIC_VERIFY_EPS or abs(val_metrics['auc']-best_auc)>METRIC_VERIFY_EPS:
        raise RuntimeError('Best checkpoint does not reproduce saved Tune-Val metrics')
    make_prediction_frame(val_keys,val_y,val_codes,val_logits,locked).to_csv(folder/'val_predictions.csv',index=False)
    pd.DataFrame(history).to_csv(folder/'training_history.csv',index=False)
    if csmr_lambda is not None:
        pd.DataFrame(diagnostics).to_csv(folder/'csmr_diagnostics.csv',index=False)
        pd.DataFrame(class_diagnostics).to_csv(folder/'csmr_class_diagnostics.csv',index=False)
    # LCC is opened only after the best Tune-Val checkpoint and threshold are frozen.
    lcc_ds=FinalLCCDataset(lcc_frame,lcc_root,lcc_cache)
    lcc_loader=DataLoader(lcc_ds,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,
        pin_memory=device.type=='cuda')
    lcc_logits,lcc_y,lcc_keys,lcc_predictions=eval_lcc_spatial(model,lcc_loader,device,use_amp,lcc_frame,locked)
    lcc_scores=binary_score(torch.from_numpy(lcc_logits)).numpy()
    lcc_metrics=metrics_binary(lcc_y,lcc_scores,locked,
        coverage=len(lcc_y)/len(lcc_frame),candidate_n=len(lcc_frame))
    lcc_predictions.to_csv(folder/'lcc_predictions.csv',index=False)
    metrics={'schema':'minifasnetv2_5k_candidate_result_v1','config_hash':conf_hash,
        'resource_hashes':resource_hashes,'initialization_sha256':initial_sha,
        'official_checkpoint_sha256':resource_hashes['official_checkpoint_sha256'] if initialization=='pretrained' else None,
        'best_epoch':best_epoch,'best_val_acer':best_acer,'best_val_auc':best_auc,
        'locked_val_threshold':locked,'threshold_source':'CelebA Tune-Val-2K only',
        'checkpoint_sha256':sha256_file(best_path),'val':val_metrics,'lcc':lcc_metrics}
    strict_json_write(metrics_path,metrics)
    result={**metrics,'stage':stage,'candidate_name':candidate_name,
        'candidate_dir':str(folder),'reused_or_new':'NEW_RUN'}
    run_registry[conf_hash]=result
    model.cpu()
    if device.type=='cuda':torch.cuda.empty_cache()
    print('FROZEN',stage,candidate_name,'Val ACER',best_acer,'Val AUC',best_auc,
          'LCC AUC',lcc_metrics['auc'],'LCC HTER',lcc_metrics['hter'])
    return result


In [28]:
# Stage A-D were already run. Reload exact candidate metrics/configs; no training in these stages.
def load_historical_candidate(stage,candidate_name,initialization,backbone_lr,classifier_lr,csmr_lambda=None,gain_range=None):
    folder=output/stage/candidate_name
    initial_sha=scratch_sha if initialization=='scratch' else pretrained_sha
    expected=candidate_config(initialization,initial_sha,backbone_lr,classifier_lr,csmr_lambda,gain_range)
    config_hash=sha256_text(json.dumps(expected,sort_keys=True,separators=(',',':')))
    config_path=folder/'config.json';metrics_path=folder/'metrics.json';best_path=folder/'best.pth'
    saved_config=strict_json_load(config_path)
    if saved_config!={**expected,'config_hash':config_hash}:
        raise RuntimeError('Historical candidate config mismatch: '+str(folder))
    if not metrics_path.is_file():
        reuse=strict_json_load(folder/'reuse_reference.json')
        if reuse['config_hash']!=config_hash or config_hash not in run_registry:
            raise RuntimeError('Historical candidate reuse mismatch: '+str(folder))
        return {**run_registry[config_hash],'stage':stage,'candidate_name':candidate_name,
                'candidate_dir':str(folder),'reused_or_new':'REUSED'}
    saved=strict_json_load(metrics_path)
    if (saved['config_hash']!=config_hash or saved['resource_hashes']!=resource_hashes or
        saved['initialization_sha256']!=initial_sha or
        saved['official_checkpoint_sha256']!=(resource_hashes['official_checkpoint_sha256'] if initialization=='pretrained' else None)):
        raise RuntimeError('Historical candidate metrics provenance mismatch: '+str(folder))
    if best_path.is_file() and sha256_file(best_path)!=saved['checkpoint_sha256']:
        raise RuntimeError('Historical best checkpoint SHA256 mismatch: '+str(best_path))
    result={**saved,'stage':stage,'candidate_name':candidate_name,
            'candidate_dir':str(folder),'reused_or_new':'REUSED'}
    run_registry[config_hash]=result
    return result

def require_historical_winner(result):
    path=Path(result['candidate_dir'])/'best.pth'
    if not path.is_file() or sha256_file(path)!=result['checkpoint_sha256']:
        raise RuntimeError('Selected Stage A-D checkpoint missing/mismatched: '+str(path))


In [30]:
def best_by_val(results):
    if not results:raise ValueError('No optimizer candidates')
    best=results[0]
    for candidate in results[1:]:
        a=candidate['val'];b=best['val']
        if (a['acer']<b['acer']-METRIC_TIE_EPS or
            (abs(a['acer']-b['acer'])<=METRIC_TIE_EPS and a['auc']>b['auc']+METRIC_TIE_EPS)):
            best=candidate
    return best

def metric_row(stage,result,backbone_lr,classifier_lr,csmr_lambda=None,gain_range=None,
               reference_acer=None,eligible=None,selected=False):
    val=result['val'];lcc=result['lcc']
    return {'stage':stage,'candidate_name':result['candidate_name'],
        'initialization':'pretrained' if stage in ('B','E','F') else 'scratch',
        'backbone_lr':backbone_lr,'classifier_lr':classifier_lr,
        'csmr_lambda':csmr_lambda,'gain_min':gain_range[0] if gain_range else None,
        'gain_max':gain_range[1] if gain_range else None,'best_epoch':result['best_epoch'],
        'val_apcer':val['apcer'],'val_bpcer':val['bpcer'],'val_acer':val['acer'],
        'val_auc':val['auc'],'val_accuracy_3class':val['accuracy_3class'],
        'lcc_apcer':lcc['apcer'],'lcc_bpcer':lcc['bpcer'],'lcc_hter':lcc['hter'],
        'lcc_auc':lcc['auc'],'lcc_detector_coverage':lcc['detector_coverage'],
        'val_acer_delta_vs_scratch_vanilla':val['acer']-reference_acer if reference_acer is not None else None,
        'eligible':eligible,'selected':selected,'reused_or_new':result['reused_or_new'],
        'initialization_sha256':result['initialization_sha256'],
        'official_checkpoint_sha256':result['official_checkpoint_sha256'],
        'candidate_dir':result['candidate_dir']}

stage_a=[]
for name,lr in zip(('lr_1e-4','lr_3e-4','lr_1e-3'),SCRATCH_LR_CANDIDATES):
    stage_a.append(load_historical_candidate('stage_a_scratch_lr',name,'scratch',lr,lr))
assert len(stage_a)==3 and {r['initialization_sha256'] for r in stage_a}=={scratch_sha}
scratch_reference=best_by_val(stage_a)
require_historical_winner(scratch_reference)
selected_scratch_lr=float(strict_json_load(Path(scratch_reference['candidate_dir'])/'config.json')['backbone_lr'])
a_rows=[metric_row('A',r,
    strict_json_load(Path(r['candidate_dir'])/'config.json')['backbone_lr'],
    strict_json_load(Path(r['candidate_dir'])/'config.json')['classifier_lr'],
    reference_acer=scratch_reference['val']['acer'],selected=r['config_hash']==scratch_reference['config_hash']) for r in stage_a]
pd.DataFrame(a_rows).to_csv(selection_dir/'stage_a_scratch_lr_results.csv',index=False)
print('Stage A selected scratch LR from Tune-Val only:',selected_scratch_lr)

# Stage A is complete before the official checkpoint is loaded for Stage B.
pretrained_model,loading_audit=load_pretrained_canonical(module,MINIFASNET_CHECKPOINT)
pretrained_state={k:v.detach().cpu().clone() for k,v in pretrained_model.state_dict().items()}
pretrained_sha=state_fingerprint(pretrained_state)
assert loading_audit['intentionally_skipped_keys']==['prob.weight']
assert loading_audit['checkpoint_sha256']==resource_hashes['official_checkpoint_sha256']
strict_json_write(output/'stage_b_pretrained_loading_audit.json',loading_audit)
init_path=tuning_dir/'initialization_fingerprints.json'
initial_fingerprints=strict_json_load(init_path)
initial_fingerprints['pretrained_sha256']=pretrained_sha
strict_json_write(init_path,initial_fingerprints)
del pretrained_model
stage_b=[]
for i,recipe in enumerate(PRETRAINED_LR_CANDIDATES):
    stage_b.append(load_historical_candidate('stage_b_pretrained_lr',f'candidate_{i}','pretrained',
        recipe['backbone_lr'],recipe['head_lr']))
assert len(stage_b)==3 and {r['initialization_sha256'] for r in stage_b}=={pretrained_sha}
pretrained_winner=best_by_val(stage_b)
require_historical_winner(pretrained_winner)
pretrained_config=strict_json_load(Path(pretrained_winner['candidate_dir'])/'config.json')
selected_pretrained_backbone_lr=float(pretrained_config['backbone_lr'])
selected_pretrained_head_lr=float(pretrained_config['classifier_lr'])
b_rows=[metric_row('B',r,
    strict_json_load(Path(r['candidate_dir'])/'config.json')['backbone_lr'],
    strict_json_load(Path(r['candidate_dir'])/'config.json')['classifier_lr'],
    reference_acer=scratch_reference['val']['acer'],selected=r['config_hash']==pretrained_winner['config_hash']) for r in stage_b]
pd.DataFrame(b_rows).to_csv(selection_dir/'stage_b_pretrained_lr_results.csv',index=False)
optimizer_selection={'schema':'minifasnetv2_5k_optimizer_selection_v1',
    'selection_policy':'minimum Tune-Val ACER, tie maximum Tune-Val AUC; LCC diagnostic only',
    'scratch_lr':selected_scratch_lr,'scratch_reference_dir':scratch_reference['candidate_dir'],
    'scratch_reference_checkpoint_sha256':scratch_reference['checkpoint_sha256'],
    'pretrained_backbone_lr':selected_pretrained_backbone_lr,
    'pretrained_head_lr':selected_pretrained_head_lr,
    'pretrained_checkpoint_sha256':resource_hashes['official_checkpoint_sha256'],
    'tune_manifest_fingerprints':tune_fp}
selection_path = selection_dir / 'optimizer_selection_before_csmr.json'

if selection_path.exists():
    previous_selection = strict_json_load(selection_path)

    # Absolute output root may legitimately change when v1 artifacts
    # are copied into the v2 output tree.
    previous_compare = dict(previous_selection)
    current_compare = dict(optimizer_selection)

    previous_dir = Path(previous_compare.pop('scratch_reference_dir'))
    current_dir = Path(current_compare.pop('scratch_reference_dir'))

    # Selection/provenance must be identical apart from relocation.
    if previous_compare != current_compare:
        diff_keys = sorted(
            k for k in set(previous_compare) | set(current_compare)
            if previous_compare.get(k) != current_compare.get(k)
        )
        raise RuntimeError(
            'Previously frozen optimizer selection differs in semantic fields: '
            + str(diff_keys)
        )

    # Even the relocated path must refer to the same Stage-A candidate.
    if previous_dir.parts[-2:] != current_dir.parts[-2:]:
        raise RuntimeError(
            'Historical scratch winner differs after output relocation: '
            f'{previous_dir} vs {current_dir}'
        )

strict_json_write(selection_path, optimizer_selection)
print('Optimizer recipes FROZEN before CSMR:', optimizer_selection)


Stage A selected scratch LR from Tune-Val only: 0.001
Optimizer recipes FROZEN before CSMR: {'schema': 'minifasnetv2_5k_optimizer_selection_v1', 'selection_policy': 'minimum Tune-Val ACER, tie maximum Tune-Val AUC; LCC diagnostic only', 'scratch_lr': 0.001, 'scratch_reference_dir': '/kaggle/working/minifasnetv2_5k_hparam_tuning_v2/stage_a_scratch_lr/lr_1e-3', 'scratch_reference_checkpoint_sha256': 'c227f349d7f320603414f9ec328270a2bb9c8f4432e240b7eea05d864ede42f5', 'pretrained_backbone_lr': 0.0002, 'pretrained_head_lr': 0.001, 'pretrained_checkpoint_sha256': 'a5eb02e1843f19b5386b953cc4c9f011c3f985d0ee2bb9819eea9a142099bec0', 'tune_manifest_fingerprints': {'tune_train5k': {'sample_count': 5000, 'class_counts': {'1': 2315, '0': 1637, '2': 1048}, 'ordered_sample_key_sha256': '52425728967f3a4e62b2915595a45a53886f07d646389a084884c65eefe7b659', 'source_manifest_sha256': '3ec46d1cc6b9bb384f244ba4fcc3657fd1e6f050b13b00b37818a913b590c8b6', 'sampling_seed': 43, 'file_sha256': '33142055672456ea95d

In [33]:
reference_acer=float(scratch_reference['val']['acer'])
regression_limit=reference_acer+VAL_ACER_MAX_REGRESSION_PP/100.
def eligible_csmr(result):return result['val']['acer']<=regression_limit+METRIC_TIE_EPS

def choose_csmr(results,parameter):
    allowed=[(result,value) for result,value in results if eligible_csmr(result)]
    if not allowed:return None
    if parameter=='lambda':
        return min(allowed,key=lambda item:(-item[0]['lcc']['auc'],item[0]['lcc']['hter'],
            item[0]['val']['acer'],float(item[1])))[0]
    # Larger gains mean less attenuation and win only after all metric ties.
    return min(allowed,key=lambda item:(-item[0]['lcc']['auc'],item[0]['lcc']['hter'],
        item[0]['val']['acer'],-float(item[1][0]),-float(item[1][1])))[0]

stage_c=[]
for name,lam in zip(('lambda_0p10','lambda_0p20','lambda_0p40'),CSMR_LAMBDA_CANDIDATES):
    result=load_historical_candidate('stage_c_lambda',name,'scratch',selected_scratch_lr,
        selected_scratch_lr,csmr_lambda=lam,gain_range=[.65,.90])
    stage_c.append((result,lam))
assert len(stage_c)==3 and {r['initialization_sha256'] for r,_ in stage_c}=={scratch_sha}
lambda_winner=choose_csmr(stage_c,'lambda')
c_rows=[metric_row('C',r,selected_scratch_lr,selected_scratch_lr,lam,[.65,.90],
    reference_acer,eligible_csmr(r),lambda_winner is not None and r['config_hash']==lambda_winner['config_hash'])
    for r,lam in stage_c]
pd.DataFrame(c_rows).to_csv(selection_dir/'stage_c_lambda_results.csv',index=False)
if lambda_winner is None:
    strict_json_write(selection_dir/'no_lambda_passed.json',{'reference_val_acer':reference_acer,
        'max_regression_pp':VAL_ACER_MAX_REGRESSION_PP,'results':c_rows})
    raise RuntimeError('No CSMR lambda passed the 1.0 pp Tune-Val ACER regression guard; no frozen hyperparameters')
selected_lambda=float(strict_json_load(Path(lambda_winner['candidate_dir'])/'config.json')['csmr']['lambda'])
print('Stage C selected lambda:',selected_lambda,'Val guard:',regression_limit)

stage_d=[]
for name,gain in CSMR_GAIN_CANDIDATES:
    result=load_historical_candidate('stage_d_gain',name,'scratch',selected_scratch_lr,
        selected_scratch_lr,csmr_lambda=selected_lambda,gain_range=gain)
    stage_d.append((result,gain))
assert len(stage_d)==3 and {r['initialization_sha256'] for r,_ in stage_d}=={scratch_sha}
assert any(r['reused_or_new']=='REUSED' for r,g in stage_d if g==[.65,.90])
gain_winner=choose_csmr(stage_d,'gain')
d_rows=[metric_row('D',r,selected_scratch_lr,selected_scratch_lr,selected_lambda,gain,
    reference_acer,eligible_csmr(r),gain_winner is not None and r['config_hash']==gain_winner['config_hash'])
    for r,gain in stage_d]
pd.DataFrame(d_rows).to_csv(selection_dir/'stage_d_gain_results.csv',index=False)
if gain_winner is None:
    strict_json_write(selection_dir/'no_gain_passed.json',{'reference_val_acer':reference_acer,
        'max_regression_pp':VAL_ACER_MAX_REGRESSION_PP,'results':d_rows})
    raise RuntimeError('No CSMR gain passed the 1.0 pp Tune-Val ACER regression guard; no frozen hyperparameters')
require_historical_winner(gain_winner)
selected_gain=list(strict_json_load(Path(gain_winner['candidate_dir'])/'config.json')['csmr']['gain_range'])
assert selected_lambda in CSMR_LAMBDA_CANDIDATES and selected_gain in [g for _,g in CSMR_GAIN_CANDIDATES]
print('Stage D selected gain range:',selected_gain)


Stage C selected lambda: 0.2 Val guard: 0.12085402148664343
Stage D selected gain range: [0.55, 0.85]


In [34]:
# Historical v1 frozen config is immutable provenance for Stages A-D.
frozen_path=selection_dir/'frozen_hparams.json'
if sha256_file(frozen_path)!=EXPECTED_V1_SHA256:
    raise RuntimeError('Historical v1 frozen config changed')
v1_frozen=strict_json_load(frozen_path)
if (v1_frozen['scratch']['backbone_lr']!=selected_scratch_lr or
    v1_frozen['pretrained']['backbone_lr']!=selected_pretrained_backbone_lr or
    v1_frozen['pretrained']['classifier_lr']!=selected_pretrained_head_lr or
    v1_frozen['csmr']['lambda']!=selected_lambda or
    v1_frozen['csmr']['gain_range']!=selected_gain):
    raise RuntimeError('Reloaded Stage A-D winners differ from historical v1 freeze')
if v1_frozen['resource_hashes']!=resource_hashes:
    raise RuntimeError('Historical v1 resource fingerprints differ')
print('Stage A-D winners match immutable v1 frozen config',EXPECTED_V1_SHA256)


Stage A-D winners match immutable v1 frozen config 2aedb0477c6d0ed05c9a96ba8918497b312fdebd0f820c1170e4c2b56398155e


In [35]:
# Stage E — pretrained lambda, Stage F — pretrained gain; same grids and selection policy as C/D.
pretrained_reference_acer=float(pretrained_winner['val']['acer'])
pretrained_regression_limit=pretrained_reference_acer+VAL_ACER_MAX_REGRESSION_PP/100.
def eligible_pretrained_csmr(result):
    return result['val']['acer']<=pretrained_regression_limit+METRIC_TIE_EPS

def choose_pretrained_csmr(results,parameter):
    allowed=[(result,value) for result,value in results if eligible_pretrained_csmr(result)]
    if not allowed:return None
    if parameter=='lambda':
        return min(allowed,key=lambda item:(-item[0]['lcc']['auc'],item[0]['lcc']['hter'],
            item[0]['val']['acer'],float(item[1])))[0]
    return min(allowed,key=lambda item:(-item[0]['lcc']['auc'],item[0]['lcc']['hter'],
        item[0]['val']['acer'],-float(item[1][0]),-float(item[1][1])))[0]

stage_e=[]
for name,lam in zip(('lambda_0p10','lambda_0p20','lambda_0p40'),PRETRAINED_CSMR_LAMBDA_CANDIDATES):
    result=run_candidate('stage_e_pretrained_lambda',name,'pretrained',
        selected_pretrained_backbone_lr,selected_pretrained_head_lr,
        csmr_lambda=lam,gain_range=[.65,.90])
    stage_e.append((result,lam))
assert len(stage_e)==3 and {r['initialization_sha256'] for r,_ in stage_e}=={pretrained_sha}
pretrained_lambda_winner=choose_pretrained_csmr(stage_e,'lambda')
e_rows=[metric_row('E',r,selected_pretrained_backbone_lr,selected_pretrained_head_lr,
    lam,[.65,.90],pretrained_reference_acer,eligible_pretrained_csmr(r),
    pretrained_lambda_winner is not None and r['config_hash']==pretrained_lambda_winner['config_hash'])
    for r,lam in stage_e]
pd.DataFrame(e_rows).to_csv(selection_dir/'stage_e_pretrained_lambda_results.csv',index=False)
if pretrained_lambda_winner is None:
    strict_json_write(selection_dir/'no_pretrained_lambda_passed.json',
        {'reference_val_acer':pretrained_reference_acer,'max_regression_pp':VAL_ACER_MAX_REGRESSION_PP,'results':e_rows})
    raise RuntimeError('No pretrained CSMR lambda passed the 1.0 pp Tune-Val ACER guard')
selected_pretrained_lambda=float(strict_json_load(Path(pretrained_lambda_winner['candidate_dir'])/'config.json')['csmr']['lambda'])

stage_f=[]
for name,gain in PRETRAINED_CSMR_GAIN_CANDIDATES:
    result=run_candidate('stage_f_pretrained_gain',name,'pretrained',
        selected_pretrained_backbone_lr,selected_pretrained_head_lr,
        csmr_lambda=selected_pretrained_lambda,gain_range=gain)
    stage_f.append((result,gain))
assert len(stage_f)==3 and {r['initialization_sha256'] for r,_ in stage_f}=={pretrained_sha}
assert any(r['reused_or_new']=='REUSED' for r,g in stage_f if g==[.65,.90])
pretrained_gain_winner=choose_pretrained_csmr(stage_f,'gain')
f_rows=[metric_row('F',r,selected_pretrained_backbone_lr,selected_pretrained_head_lr,
    selected_pretrained_lambda,gain,pretrained_reference_acer,eligible_pretrained_csmr(r),
    pretrained_gain_winner is not None and r['config_hash']==pretrained_gain_winner['config_hash'])
    for r,gain in stage_f]
pd.DataFrame(f_rows).to_csv(selection_dir/'stage_f_pretrained_gain_results.csv',index=False)
if pretrained_gain_winner is None:
    strict_json_write(selection_dir/'no_pretrained_gain_passed.json',
        {'reference_val_acer':pretrained_reference_acer,'max_regression_pp':VAL_ACER_MAX_REGRESSION_PP,'results':f_rows})
    raise RuntimeError('No pretrained CSMR gain passed the 1.0 pp Tune-Val ACER guard')
selected_pretrained_gain=list(strict_json_load(Path(pretrained_gain_winner['candidate_dir'])/'config.json')['csmr']['gain_range'])
assert selected_pretrained_lambda in PRETRAINED_CSMR_LAMBDA_CANDIDATES
assert selected_pretrained_gain in [g for _,g in PRETRAINED_CSMR_GAIN_CANDIDATES]
print('Pretrained CSMR recipe:',selected_pretrained_lambda,selected_pretrained_gain)


stage_e_pretrained_lambda/lambda_0p10 epoch 1/6


/tmp/ipykernel_58/81287471.py:65: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler=torch.cuda.amp.GradScaler(enabled=use_amp)
/tmp/ipykernel_58/81287471.py:121: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


  batch 1 / 40 mean loss 1.1446527242660522
  batch 5 / 40 mean loss 0.9999795436859131
  batch 10 / 40 mean loss 0.88411585688591
  batch 15 / 40 mean loss 0.8009970545768738
  batch 20 / 40 mean loss 0.7347901612520218
  batch 25 / 40 mean loss 0.6882226133346557
  batch 30 / 40 mean loss 0.6617419322331747
  batch 35 / 40 mean loss 0.6347722343036106
  batch 40 / 40 mean loss 0.6119081646203994


/tmp/ipykernel_58/1261348889.py:165: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=amp):


  Tune-Val ACER 0.4664997096399536 AUC 0.540010707026713 best epoch 1
stage_e_pretrained_lambda/lambda_0p10 epoch 2/6


...

/tmp/ipykernel_58/1261348889.py:165: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=amp):


  Tune-Val ACER 0.03747459349593496 AUC 0.9873942907955865 best epoch 6


/tmp/ipykernel_58/1261348889.py:165: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=amp):
/tmp/ipykernel_58/1261348889.py:187: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=amp):


FROZEN stage_f_pretrained_gain strong Val ACER 0.03747459349593496 Val AUC 0.9873942907955865 LCC AUC 0.7791939163034439 LCC HTER 0.35876158546205045
Pretrained CSMR recipe: 0.4 [0.65, 0.9]


/tmp/ipykernel_58/1261348889.py:217: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[ True  True  True ... False  True  True]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  full.loc[valid, "pred_real"] = full.loc[valid, "pad_logit_score"] >= float(threshold)


In [36]:
# Freeze regime-specific recipes only after Stage F succeeds.
legacy_rows=a_rows+b_rows+c_rows+d_rows
all_rows=[]
for row in legacy_rows+e_rows+f_rows:
    item=dict(row)
    item['regime']=item['initialization']
    item['val_acer_delta_vs_regime_vanilla']=item['val_acer']-(
        scratch_reference['val']['acer'] if item['regime']=='scratch' else pretrained_winner['val']['acer'])
    candidate_metrics=strict_json_load(Path(item['candidate_dir'])/'metrics.json') if (Path(item['candidate_dir'])/'metrics.json').is_file() else None
    item['checkpoint_sha256']=candidate_metrics['checkpoint_sha256'] if candidate_metrics else run_registry[
        strict_json_load(Path(item['candidate_dir'])/'config.json')['config_hash']]['checkpoint_sha256']
    all_rows.append(item)
summary_v2=pd.DataFrame(all_rows)
assert len(summary_v2)==18 and summary_v2.stage.value_counts().to_dict()=={s:3 for s in 'ABCDEF'}
summary_v2.to_csv(selection_dir/'tuning_summary_v2.csv',index=False)
optimizer_common={'name':'AdamW','weight_decay':WEIGHT_DECAY,
    'label_smoothing':LABEL_SMOOTHING,'grad_clip':GRAD_CLIP}
frozen_v2={'schema':'minifasnetv2_final_hparams_v2','selection_stage':'5k_regime_specific_tuning',
    'data':v1_frozen['data'],
    'scratch':{'optimizer':{**optimizer_common,'backbone_lr':selected_scratch_lr,
        'classifier_lr':selected_scratch_lr},
        'csmr':{'lambda':selected_lambda,'gain_range':selected_gain}},
    'pretrained':{'optimizer':{**optimizer_common,'backbone_lr':selected_pretrained_backbone_lr,
        'classifier_lr':selected_pretrained_head_lr},
        'csmr':{'lambda':selected_pretrained_lambda,'gain_range':selected_pretrained_gain},
        'checkpoint_sha256':resource_hashes['official_checkpoint_sha256']},
    'csmr_structure':{'band_centers':CSMR_BAND_CENTERS,'band_sigma':CSMR_BAND_SIGMA,
        'warmup_epochs':CSMR_WARMUP_EPOCHS,'preserve_dc':CSMR_PRESERVE_DC,
        'clean_ce_weight':.5,'worst_ce_weight':.5},
    'selection_policy':{'optimizer':'Tune-Val ACER then Tune-Val AUC',
        'csmr_source_guard':'Val ACER regression <= 1.0 pp',
        'csmr_primary_external_dev':'LCC AUC','csmr_secondary_external_dev':'LCC HTER',
        'lcc_role':'external-development; not untouched final test'},
    'training_seed':TRAIN_SEED,'batch_size':BATCH_SIZE,'max_epochs':MAX_EPOCHS,
    'resource_hashes':resource_hashes,
    'initialization_sha256':{'scratch':scratch_sha,'pretrained':pretrained_sha},
    'selected_run_checkpoint_sha256':{
        'scratch_vanilla':scratch_reference['checkpoint_sha256'],
        'pretrained_vanilla':pretrained_winner['checkpoint_sha256'],
        'scratch_csmr':gain_winner['checkpoint_sha256'],
        'pretrained_csmr':pretrained_gain_winner['checkpoint_sha256']},
    'historical_v1_frozen_sha256':EXPECTED_V1_SHA256}
v2_path=selection_dir/'frozen_hparams_v2.json'
if v2_path.exists() and strict_json_load(v2_path)!=frozen_v2:
    raise RuntimeError('Existing v2 frozen config differs; stop')
strict_json_write(v2_path,frozen_v2)
(selection_dir/'frozen_hparams_v2.sha256').write_text(sha256_file(v2_path)+'  frozen_hparams_v2.json\n')

def stage_table(rows):
    cols=['candidate_name','backbone_lr','classifier_lr','csmr_lambda','gain_min','gain_max',
          'val_acer','val_auc','lcc_hter','lcc_auc','eligible','selected','reused_or_new']
    return '```\n'+pd.DataFrame(rows)[cols].to_string(index=False)+'\n```'
report=['# MiniFASNetV2 5K regime-specific hyperparameter tuning','',
    '## Objective','Freeze separate scratch and pretrained CSMR recipes under symmetric search budgets. One 5K/2K split and one seed are screening evidence only.','',
    '## Frozen resources',f'Tune-Train-5K and Tune-Val-2K fingerprints: `{tune_fp}`. Resource hashes: `{resource_hashes}`.','',
    '## Stage A — scratch Vanilla LR',stage_table(a_rows),
    f'Selected scratch LR {selected_scratch_lr} by Tune-Val ACER then AUC only.','',
    '## Stage B — pretrained Vanilla LR',stage_table(b_rows),
    f'Selected backbone/head LR {selected_pretrained_backbone_lr}/{selected_pretrained_head_lr} by Tune-Val only. Official feature weights and deterministic fresh classifier share SHA {pretrained_sha}.','',
    '## Scratch Vanilla reference',f'Val ACER {scratch_reference["val"]["acer"]:.6f}, LCC AUC {scratch_reference["lcc"]["auc"]:.6f}.','',
    '## Pretrained Vanilla reference',f'Val ACER {pretrained_reference_acer:.6f}, LCC AUC {pretrained_winner["lcc"]["auc"]:.6f}.','',
    '## Stage C — scratch lambda',stage_table(c_rows),'',
    '## Stage D — scratch gain',stage_table(d_rows),
    f'Selected scratch CSMR lambda {selected_lambda}, gain {selected_gain}.','',
    '## Stage E — pretrained lambda',stage_table(e_rows),'',
    '## Stage F — pretrained gain',stage_table(f_rows),
    f'Selected pretrained CSMR lambda {selected_pretrained_lambda}, gain {selected_pretrained_gain}. Current-gain Stage-F candidate reuses Stage E.','',
    '## Regime comparison',
    f'Scratch selected ({selected_lambda}, {selected_gain}); pretrained selected ({selected_pretrained_lambda}, {selected_pretrained_gain}). These recipes were selected with separate Vanilla Val ACER guards and the same LCC ranking rule.','',
    '## Evaluation boundary','LCC-FASD was used as an external-development robustness signal for CSMR selection; it is not an untouched final external test. Test-5K was never evaluated in 00b-v2. CASIA-FASD remains untouched. No 20K result may trigger retuning.','',
    '## Provenance',f'Historical v1 frozen_hparams.json SHA256: `{EXPECTED_V1_SHA256}` (scratch CSMR only). New frozen_hparams_v2.json SHA256: `{sha256_file(v2_path)}` (regime-specific CSMR).','',
    '## Next experiments','01 scratch 20K; 01b pretrained 20K; 02 scratch 100K; 03 pretrained 100K. Hard-code these frozen v2 recipes in those notebooks after downloading this output; do not select again.']
(selection_dir/'tuning_report_v2.md').write_text('\n'.join(report)+'\n')
print('FROZEN v2:',v2_path,'SHA256:',sha256_file(v2_path))
print(summary_v2[['stage','regime','candidate_name','val_acer','lcc_auc','eligible','selected','reused_or_new']].to_string(index=False))


FROZEN v2: /kaggle/working/minifasnetv2_5k_hparam_tuning_v2/selection/frozen_hparams_v2.json SHA256: bfc8652b237d7a8156bf455667bcbc00794baa9b0f0283c4a449406da6caeb0a
stage     regime candidate_name  val_acer  lcc_auc eligible  selected reused_or_new
    A    scratch        lr_1e-4  0.141369 0.636794     None     False        REUSED
    A    scratch        lr_3e-4  0.130099 0.630511     None     False        REUSED
    A    scratch        lr_1e-3  0.110854 0.631663     None      True        REUSED
    B pretrained    candidate_0  0.024544 0.816983     None     False        REUSED
    B pretrained    candidate_1  0.026704 0.820519     None     False        REUSED
    B pretrained    candidate_2  0.022258 0.789718     None      True        REUSED
    C    scratch    lambda_0p10  0.103477 0.591472     True     False        REUSED
    C    scratch    lambda_0p20  0.115246 0.617759     True      True        REUSED
    C    scratch    lambda_0p40  0.107959 0.551824     True     False        R

In [37]:
ZIP_PATH=Path('/kaggle/working/minifasnetv2_5k_hparam_tuning_v2.zip')
selected_best={Path(r['candidate_dir'])/'best.pth' for r in
    (scratch_reference,pretrained_winner,gain_winner,pretrained_gain_winner)}
for path in selected_best:
    if not path.is_file():raise RuntimeError('Selected checkpoint missing: '+str(path))
with zipfile.ZipFile(ZIP_PATH,'w',zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(output.rglob('*')):
        if not path.is_file() or path.name=='last_state.pth':continue
        if path.name=='best.pth' and path not in selected_best:continue
        archive.write(path,arcname=output.name+'/'+str(path.relative_to(output)))
with zipfile.ZipFile(ZIP_PATH) as archive:
    names=set(archive.namelist())
    for path in selected_best:
        if output.name+'/'+str(path.relative_to(output)) not in names:
            raise RuntimeError('Selected checkpoint missing from ZIP: '+str(path))
    for relative in ('selection/frozen_hparams.json','selection/frozen_hparams_v2.json',
        'selection/frozen_hparams_v2.sha256','selection/tuning_summary_v2.csv',
        'selection/tuning_report_v2.md'):
        if output.name+'/'+relative not in names:raise RuntimeError('Missing from ZIP: '+relative)
print('ZIP:',ZIP_PATH,'size MB:',ZIP_PATH.stat().st_size/1e6)
display(FileLink(str(ZIP_PATH)))


ZIP: /kaggle/working/minifasnetv2_5k_hparam_tuning_v2.zip size MB: 9.106346


/kaggle/working/minifasnetv2_5k_hparam_tuning_v2.zip